In [ ]:
# =============================================================================
# PIPELINE — PREDICCIÓN DE ANEMIA INFANTIL (ENDES 2024 PERÚ)
# VERSION 3 MODELOS: Random Forest, Regresion Logistica, Arbol de Decision
# Réplica del estudio de Etiopía (Yimer et al. 2025) con correcciones
#
# PASOS 01-04: Carga, Control de calidad, Merges, Ingeniería de variables
# PASOS 05-06: Modelado (solo 3 algoritmos) y evaluacion
# Archivo completo: ejecutar los pasos 01-04 en una celda y 05-06 en otra.
#
# CORRECCIONES INCORPORADAS:
#  - CASEID con .strip() antes de [:9]  (padding de 6 espacios)
#  - REC41: último nacimiento = MIDX máximo por hogar
#  - HA56 (hemoglobina materna continua) reemplaza a HA57A (sin colinealidad)
#  - M14 recodificada (Opción 3): control prenatal temprano + categoría "sin control"
#  - Coerción numérica para limpiar celdas ' ' antes de imputar
#  - 29 predictores finales
# =============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("=" * 90)
print("PIPELINE — ANEMIA INFANTIL ENDES 2024 — PASOS 01 A 04")
print("=" * 90)

# =============================================================================
# PASO 01 — CARGA DE MÓDULOS
# =============================================================================
print("\n" + "=" * 90)
print("PASO 01 — CARGA DE MÓDULOS")
print("=" * 90)

# --- Boton de carga de archivos (Colab) ---
# Aparece un boton "Choose Files": selecciona los 9 CSV de ENDES 2024.
from google.colab import files
import glob

print("\nSube los 9 archivos CSV de ENDES 2024:")
files.upload()

# Busqueda flexible: tolera renombrados de Colab tipo 'RECH6_2024 (1).csv'
def cargar(nombre):
    patron = glob.glob(f'/content/{nombre}_2024*.csv')
    if not patron:
        raise FileNotFoundError(f"No se encontro {nombre}_2024.csv. Subelo de nuevo.")
    return pd.read_csv(max(patron))   # usa el mas reciente si hay duplicados

RECH0  = cargar('RECH0')
RECH4  = cargar('RECH4')
RECH5  = cargar('RECH5')
RECH6  = cargar('RECH6')
RECH23 = cargar('RECH23')
REC41  = cargar('REC41')
REC43  = cargar('REC43')
REC94  = cargar('REC94')
REC95  = cargar('REC95')

for nombre, df in [('RECH0',RECH0),('RECH4',RECH4),('RECH5',RECH5),('RECH6',RECH6),
                    ('RECH23',RECH23),('REC41',REC41),('REC43',REC43),
                    ('REC94',REC94),('REC95',REC95)]:
    print(f"  {nombre:8} {df.shape[0]:>7,} filas x {df.shape[1]:>3} columnas")

print("\n  9/9 modulos cargados correctamente")

# =============================================================================
# PASO 02 — CONTROL DE CALIDAD
# =============================================================================
print("\n" + "=" * 90)
print("PASO 02 — CONTROL DE CALIDAD")
print("=" * 90)

print(f"\n  RECH6 original: {len(RECH6):,} ninos <5 anios")

# Filtro de calidad sobre RECH6 (segun decisiones documentadas)
df = RECH6.copy()

# Filtro 1: HC57A != 9 (excluir anemia no medida)
n0 = len(df)
df = df[df['HC57A'] != 9].copy()
print(f"  Filtro HC57A != 9 : -{n0-len(df):,}  -> {len(df):,}")

# Filtro 2: HC70 plausible (z-score talla/edad valido, <9996)
n1 = len(df)
df['HC70'] = pd.to_numeric(df['HC70'], errors='coerce')
df = df[df['HC70'] < 9996].copy()
print(f"  Filtro HC70 < 9996: -{n1-len(df):,}  -> {len(df):,}")

# Filtro 3: HC55 == 0 (solo hemoglobina efectivamente medida)
n2 = len(df)
df = df[df['HC55'] == 0].copy()
print(f"  Filtro HC55 == 0  : -{n2-len(df):,}  -> {len(df):,}")

# Target binario
df['target'] = df['HC57A'].isin([1, 2, 3]).astype(int)

n_pos = int(df['target'].sum())
n_neg = int((df['target'] == 0).sum())
print(f"\n  TARGET:")
print(f"    Sin anemia (0): {n_neg:>7,} ({100*n_neg/len(df):.1f}%)")
print(f"    Con anemia (1): {n_pos:>7,} ({100*n_pos/len(df):.1f}%)")

df_elegibles = df.copy()
print(f"\n  df_elegibles: {df_elegibles.shape}")

# =============================================================================
# PASO 03 — MERGES
# =============================================================================
print("\n" + "=" * 90)
print("PASO 03 — MERGES")
print("=" * 90)

df = df_elegibles.copy()

# --- CLAVES ---
# HHID: clave de hogar. Convertir a string limpio.
df['HHID'] = df['HHID'].astype(str).str.strip()
df['HC0']  = pd.to_numeric(df['HC0'], errors='coerce')
df['HC1']  = pd.to_numeric(df['HC1'], errors='coerce')

# HIDX_match: rango del nino por EDAD ascendente dentro del hogar.
# Los modulos individuales (REC43/94/95) usan HIDX = orden del nino en el hogar,
# NO HC0. La clave correcta es HHID + rank-de-edad. Esto recupera ~19,300 matches
# (frente a ~140 si se intenta un CASEID construido a mano).
df['HIDX_match'] = df.groupby('HHID')['HC1'].rank(method='first').astype('Int64')

print(f"\n  HHID unicos       : {df['HHID'].nunique():,}")
print(f"  Ninos (filas)     : {len(df):,}")
print(f"  HIDX_match        : rango por edad ascendente dentro del hogar")

# ---------------------------------------------------------------------------
# MERGE 1: RECH0 (hogar) por HHID
# ---------------------------------------------------------------------------
r0 = RECH0.copy()
r0['HHID'] = r0['HHID'].astype(str).str.strip()
cols_r0 = ['HHID', 'HV040', 'HV025', 'HV024', 'HV014', 'HV005']
r0 = r0[cols_r0].drop_duplicates('HHID')
df = df.merge(r0, on='HHID', how='left', indicator=True)
print(f"\n  Merge 1 RECH0   : {(df['_merge']=='left_only').sum()} sin match")
df = df.drop(columns='_merge')

# ---------------------------------------------------------------------------
# MERGE 2: RECH23 (riqueza y servicios) por HHID
# ---------------------------------------------------------------------------
r23 = RECH23.copy()
r23['HHID'] = r23['HHID'].astype(str).str.strip()
cols_r23 = ['HHID', 'HV270', 'HV201', 'HV205', 'HV226']
r23 = r23[cols_r23].drop_duplicates('HHID')
df = df.merge(r23, on='HHID', how='left', indicator=True)
print(f"  Merge 2 RECH23  : {(df['_merge']=='left_only').sum()} sin match")
df = df.drop(columns='_merge')

# ---------------------------------------------------------------------------
# MERGE 3: RECH5 (anemia/hemoglobina materna) por HHID
#   HA56 = hemoglobina materna continua (g/dL x10). Reemplaza a HA57A.
# ---------------------------------------------------------------------------
r5 = RECH5.copy()
r5['HHID'] = r5['HHID'].astype(str).str.strip()
r5 = r5[['HHID', 'HA56', 'HA1']].drop_duplicates('HHID')
df = df.merge(r5, on='HHID', how='left', indicator=True)
print(f"  Merge 3 RECH5   : {(df['_merge']=='left_only').sum()} sin match"
      f"  | HA56 valido: {df['HA56'].notna().sum():,}")
df = df.drop(columns='_merge')

# ---------------------------------------------------------------------------
# MERGE 4: RECH4 (madre como miembro del hogar) por HHID + HC60 = IDXH4
#   SH13  = ocupacion de la madre   -> madre_trabaja
#   SH11C = seguro SIS de la madre  -> madre_sis
# ---------------------------------------------------------------------------
r4 = RECH4.copy()
r4['HHID'] = r4['HHID'].astype(str).str.strip()
r4['IDXH4'] = pd.to_numeric(r4['IDXH4'], errors='coerce')
r4 = r4[['HHID', 'IDXH4', 'SH13', 'SH11C']]
df['HC60'] = pd.to_numeric(df['HC60'], errors='coerce')
df = df.merge(r4, left_on=['HHID', 'HC60'], right_on=['HHID', 'IDXH4'],
              how='left', indicator=True)
print(f"  Merge 4 RECH4   : {(df['_merge']=='left_only').sum()} sin match"
      f"  | SH13 valido: {df['SH13'].notna().sum():,}")
df = df.drop(columns=['_merge', 'IDXH4'])

# ---------------------------------------------------------------------------
# MERGE 5: REC41 (madre - gestacion) — ULTIMO NACIMIENTO
#   CORRECCION: CASEID tiene 6 espacios de padding -> .strip() antes de [:9]
#   CORRECCION: MIDX maximo por hogar = ultimo nacimiento
#   M13 = MES de gestacion en el 1er control prenatal (0=sin control, 1-9=mes,
#         98=no sabe). Esta es la "edad gestacional al iniciar control".
#         OJO: M14 es el NUMERO de visitas, no el momento -> no se usa.
#   M45 = hierro durante el embarazo ; M18 = tamanio al nacer
# ---------------------------------------------------------------------------
r41 = REC41.copy()
r41['HHID'] = r41['CASEID'].astype(str).str.strip().str[:9]
r41['MIDX'] = pd.to_numeric(r41['MIDX'], errors='coerce')
# ultimo nacimiento = fila con MIDX maximo dentro del hogar
idx_ultimo = r41.groupby('HHID')['MIDX'].idxmax()
r41 = r41.loc[idx_ultimo, ['HHID', 'M13', 'M45', 'M18']].copy()
df = df.merge(r41, on='HHID', how='left', indicator=True)
print(f"  Merge 5 REC41   : {(df['_merge']=='left_only').sum()} sin match"
      f"  | M13 valido: {df['M13'].notna().sum():,}")
df = df.drop(columns='_merge')

# ---------------------------------------------------------------------------
# MERGE 6: REC43 (salud del nino) por HHID + HIDX_match
#   REC43 usa HIDX = orden del nino en el hogar. Se construye HHID desde
#   CASEID (.strip() + [:9]) y se cruza con el rank de edad calculado arriba.
# ---------------------------------------------------------------------------
r43 = REC43.copy()
r43['HHID'] = r43['CASEID'].astype(str).str.strip().str[:9]
r43['HIDX'] = pd.to_numeric(r43['HIDX'], errors='coerce').astype('Int64')
r43 = r43[['HHID', 'HIDX', 'H11', 'H31', 'H31B', 'H22', 'H43']].drop_duplicates(['HHID','HIDX'])
df = df.merge(r43, left_on=['HHID','HIDX_match'], right_on=['HHID','HIDX'],
              how='left', indicator=True)
print(f"  Merge 6 REC43   : {(df['_merge']=='left_only').sum()} sin match")
df = df.drop(columns=['_merge','HIDX'])

# ---------------------------------------------------------------------------
# MERGE 7: REC94 (SIS del nino) por HHID + IDX94
# ---------------------------------------------------------------------------
r94 = REC94.copy()
r94['HHID'] = r94['CASEID'].astype(str).str.strip().str[:9]
r94['IDX94'] = pd.to_numeric(r94['IDX94'], errors='coerce').astype('Int64')
r94 = r94[['HHID', 'IDX94', 'S413']].drop_duplicates(['HHID','IDX94'])
df = df.merge(r94, left_on=['HHID','HIDX_match'], right_on=['HHID','IDX94'],
              how='left', indicator=True)
print(f"  Merge 7 REC94   : {(df['_merge']=='left_only').sum()} sin match")
df = df.drop(columns=['_merge','IDX94'])

# ---------------------------------------------------------------------------
# MERGE 8: REC95 (suplementacion Peru) por HHID + IDX95
#   S465EA = hierro jarabe 7 dias ; S465EB = micronutrientes 7 dias
#   S466   = control CRED
# ---------------------------------------------------------------------------
r95 = REC95.copy()
r95['HHID'] = r95['CASEID'].astype(str).str.strip().str[:9]
r95['IDX95'] = pd.to_numeric(r95['IDX95'], errors='coerce').astype('Int64')
r95 = r95[['HHID', 'IDX95', 'S465EA', 'S465EB', 'S466']].drop_duplicates(['HHID','IDX95'])
df = df.merge(r95, left_on=['HHID','HIDX_match'], right_on=['HHID','IDX95'],
              how='left', indicator=True)
print(f"  Merge 8 REC95   : {(df['_merge']=='left_only').sum()} sin match")
df = df.drop(columns=['_merge','IDX95'])

df_maestro = df.copy()
print(f"\n  df_maestro: {df_maestro.shape}")

# =============================================================================
# PASO 04 — INGENIERIA DE VARIABLES (29 predictores)
# =============================================================================
print("\n" + "=" * 90)
print("PASO 04 — INGENIERIA DE VARIABLES")
print("=" * 90)

d = df_maestro.copy()

# Coercion numerica general: convierte ' ' y '   ' en NaN reales
def num(serie):
    return pd.to_numeric(serie.astype(str).str.strip().replace('', np.nan), errors='coerce')

M = pd.DataFrame(index=d.index)

# --- NINO (4) ---
M['edad_meses'] = num(d['HC1'])
M['sexo_nino']  = (num(d['HC27']) == 1).astype(float)          # 1 = hombre
M['stunting']   = (num(d['HC70']) < -200).astype(float)        # z < -2.0
M['wasting']    = (num(d['HC72']) < -200).astype(float)        # z < -2.0

# --- HOGAR (5) ---
M['altitud_msnm']   = num(d['HV040'])
M['area_urbana']    = (num(d['HV025']) == 1).astype(float)     # 1 = urbano
M['region']         = num(d['HV024'])
M['nro_ninos_h']    = num(d['HV014'])
M['indice_riqueza'] = num(d['HV270'])

# --- AGUA / SANEAMIENTO (3)  [variables derivadas] ---
M['agua_mejorada'] = num(d['HV201']).isin([11, 12, 13]).astype(float)
M['san_mejorado']  = num(d['HV205']).isin([11, 12]).astype(float)
M['comb_limpio']   = num(d['HV226']).isin([1, 2, 3]).astype(float)

# --- MADRE (6) ---
M['educ_madre']    = num(d['HC61'])
M['madre_trabaja'] = num(d['SH13']).isin([1, 2, 3, 4]).astype(float)
M['madre_sis']     = (num(d['SH11C']) == 1).astype(float)
M['edad_madre']    = num(d['HA1'])
# hb_materna: HA56 es g/dL x10. 999 (y >=900) es centinela de "no medido" -> NaN.
# Rango plausible 70-200 (7.0-20.0 g/dL); fuera de rango -> NaN.
hb = num(d['HA56'])
hb = hb.where((hb >= 70) & (hb <= 200), np.nan)
M['hb_materna'] = hb
# M13 recodificada — Opcion 3: el vacio se vuelve senal.
#   M13 = mes de gestacion en el 1er control prenatal (1-9). 0 = sin control.
#   ctrl_prenatal_temprano = 1 si el control empezo en el 1er trimestre (mes 1-3)
#   sin_ctrl_prenatal      = 1 si M13 = 0 (sin control) o falta el dato
m13 = num(d['M13'])
m13 = m13.where(m13 != 98, np.nan)                              # 98 = no sabe -> NaN
M['ctrl_prenatal_temprano'] = ((m13 >= 1) & (m13 <= 3)).astype(float)
M['sin_ctrl_prenatal']      = ((m13 == 0) | (m13.isna())).astype(float)

# --- GESTACION (2) ---
M['hierro_emb']      = (num(d['M45']) == 1).astype(float)
M['bajo_peso_nacer'] = num(d['M18']).isin([4, 5]).astype(float)  # 4=mas pequenio, 5=muy pequenio

# --- SALUD INFANTIL (5) ---
M['diarrea']     = (num(d['H11'])  == 2).astype(float)         # 2 = si
tos = num(d['H31'])
M['tos']         = (tos == 2).astype(float)                    # 2 = si
# H31B: pregunta filtro (solo si hubo tos). NaN -> 0 (no tuvo tos = no IRA)
ira = (num(d['H31B']) == 1).astype(float)
M['ira']         = ira.where(tos == 2, 0.0)
M['vitamina_a']  = (num(d['H22']) == 1).astype(float)
M['antiparasit'] = (num(d['H43']) == 1).astype(float)

# --- SIS DEL NINO (1) ---
M['sis_nino'] = (num(d['S413']) == 1).astype(float)

# --- SUPLEMENTACION PERU (3) ---
M['hierro_7d']      = (num(d['S465EA']) == 1).astype(float)
M['micronut_minsa'] = (num(d['S465EB']) == 1).astype(float)
M['cred']           = (num(d['S466'])   == 1).astype(float)

# --- TARGET ---
M['target'] = d['target'].values

# --- PESO MUESTRAL (no es predictor; se usa como sample_weight en Paso 05) ---
# HV005 es el peso del hogar; se divide entre 1,000,000 (estandar DHS).
M['peso_muestral'] = pd.to_numeric(d['HV005'], errors='coerce') / 1e6

# ---------------------------------------------------------------------------
# MANEJO DE NaN
#   - hb_materna: ~4,000 madres con hemoglobina no medida (centinela 999).
#     NO se eliminan esas filas (seria perder ~22% de la muestra). En su lugar:
#       * se crea la bandera 'hb_materna_falta' (1 = dato ausente)
#       * el NaN de hb_materna se IMPUTA con la mediana DENTRO del pipeline
#         del Paso 05 (SimpleImputer), nunca aqui, para evitar leakage.
#   - Otras continuas (edad_meses, altitud, region, riqueza, educ, edad_madre,
#     nro_ninos_h): el NaN residual es minimo -> se eliminan esas pocas filas.
#   - Binarias: ya quedaron en 0/1 por construccion.
# ---------------------------------------------------------------------------
# Bandera de hemoglobina materna ausente (antes de imputar)
M['hb_materna_falta'] = M['hb_materna'].isna().astype(float)

# Continuas que SI exigen eliminar fila si faltan (NaN residual minimo)
continuas_strict = ['edad_meses', 'altitud_msnm', 'region', 'nro_ninos_h',
                    'indice_riqueza', 'educ_madre', 'edad_madre']

print(f"\n  NaN en variables continuas (antes de limpieza):")
for c in continuas_strict + ['hb_materna']:
    print(f"    {c:20}: {M[c].isna().sum():>6,}")

n_antes = len(M)
M = M.dropna(subset=continuas_strict).copy()
print(f"\n  Filas eliminadas por NaN residual: {n_antes - len(M):,}")

# hb_materna: imputar mediana AQUI es aceptable solo como marcador temporal,
# pero lo correcto es dejar el NaN y que el pipeline lo impute. Para que
# df_modelo no tenga NaN y el Paso 05 sea simple, se imputa con la mediana
# del TRAIN dentro del pipeline. Aqui se rellena con mediana global solo
# para inspeccion; el Paso 05 reimputara correctamente sobre el split.
mediana_hb = M['hb_materna'].median()
M['hb_materna'] = M['hb_materna'].fillna(mediana_hb)
print(f"  hb_materna: NaN rellenado con mediana provisional ({mediana_hb:.0f})")
print(f"              (el Paso 05 reimputa con la mediana del TRAIN)")

print(f"\n  df_modelo final: {M.shape[0]:,} ninos x {M.shape[1]} columnas")

# Verificacion final
assert M.isna().sum().sum() == 0, "Aun hay NaN en df_modelo"
PREDICTORES = [c for c in M.columns if c not in ('target', 'peso_muestral')]
print(f"\n  Predictores: {len(PREDICTORES)}")
print(f"  Target -> sin anemia: {(M['target']==0).sum():,} | "
      f"con anemia: {(M['target']==1).sum():,} "
      f"({100*M['target'].mean():.1f}% positivos)")

print(f"\n  Lista de {len(PREDICTORES)} predictores:")
for i, p in enumerate(PREDICTORES, 1):
    print(f"    {i:2}. {p}")

df_modelo = M.copy()

print("\n" + "=" * 90)
print("PASOS 01-04 COMPLETADOS  —  df_modelo listo para el PASO 05 (modelado)")
print("=" * 90)

##########################################################################################
# PIPELINE — PASOS 05 y 06  (VERSION 3 MODELOS)
# PREDICCION DE ANEMIA INFANTIL — ENDES 2024 PERU
#
# Esta version usa SOLO 3 algoritmos:
#   - Random Forest
#   - Regresion Logistica
#   - Arbol de Decision
#
# Ejecutar DESPUES del bloque de Pasos 01-04 (necesita df_modelo y PREDICTORES).
#
# Mantiene la misma logica del pipeline original:
#   - SIN SMOTE. Se usa class_weight='balanced'.
#   - hb_materna se imputa con SimpleImputer(median) DENTRO del pipeline.
#   - Doble entrenamiento: modelo SIN ponderar (comparable con Etiopia) y
#     modelo PONDERADO con peso_muestral (HV005/1e6, representativo de Peru).
#   - Verificacion de brecha CV-Test en cada modelo (detector de leakage).
# =============================================================================

import numpy as np
import pandas as pd
import time, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix)

print("=" * 90)
print("PASO 05 — MODELADO (3 algoritmos: RF, Regresion Logistica, Arbol de Decision)")
print("=" * 90)

assert 'df_modelo' in dir(), "Ejecuta primero el bloque de Pasos 01-04"

# ---------------------------------------------------------------------------
# 05.1 — Preparar X, y, peso
# ---------------------------------------------------------------------------
X = df_modelo[PREDICTORES].copy()
y = df_modelo['target'].astype(int).values
w = df_modelo['peso_muestral'].values

# split 80/20 estratificado; el peso se separa con el mismo indice
idx = np.arange(len(X))
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    X, y, idx, test_size=0.20, stratify=y, random_state=42)
wtr, wte = w[itr], w[ite]

print(f"\n  Train: {len(Xtr):,}  |  Test: {len(Xte):,}")
print(f"  Prevalencia anemia  train: {ytr.mean()*100:.1f}%  test: {yte.mean()*100:.1f}%")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------------------------------------------------------------------------
# 05.2 — Pipeline base y busqueda de hiperparametros (Grid Search)
#
#   Cada pipeline: SimpleImputer(median) -> StandardScaler -> clasificador
#   class_weight='balanced' en los tres modelos para tratar el desbalance
#   de clases (1:2.2) sin SMOTE, que introduciria fuga de informacion y
#   generaria valores decimales en las 23 variables nominales del dataset.
#
#   Grid Search acotado por conocimiento del dominio, validado con el mismo
#   Stratified 5-Fold del resto del pipeline. Se prefiere malla reducida
#   sobre busqueda exhaustiva o bayesiana porque:
#     (i)  el numero de hiperparametros relevantes es pequeno,
#     (ii) el objetivo es controlar el sobreajuste, no exprimir decimas de AUC.
#   Prefijo 'clf__' para referenciar el clasificador dentro del Pipeline.
# ---------------------------------------------------------------------------
def pipe(modelo):
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('clf', modelo),
    ])

# ── Espacios de busqueda (Tabla 2 del informe) ───────────────────────────
param_grid = {
    'Logistic Regression': {
        'clf__max_iter':     [1000, 2000, 3000],
        'clf__class_weight': ['balanced'],
    },
    'Random Forest': {
        'clf__n_estimators':     [100, 300, 500],
        'clf__max_depth':        [8, 12, 20],
        'clf__min_samples_leaf': [10, 20, 50],
        'clf__class_weight':     ['balanced'],
    },
    'Decision Tree': {
        'clf__max_depth':        [4, 6, 12],
        'clf__min_samples_leaf': [20, 50, 100],
        'clf__class_weight':     ['balanced'],
    },
}

# ── Modelos base (Grid Search sobreescribira los hiperparametros) ─────────
base_modelos = {
    'Logistic Regression': pipe(LogisticRegression(random_state=42)),
    'Random Forest':       pipe(RandomForestClassifier(random_state=42, n_jobs=-1)),
    'Decision Tree':       pipe(DecisionTreeClassifier(random_state=42)),
}

# ── Ejecutar Grid Search con Stratified 5-Fold sobre Xtr ─────────────────
print("\n" + "-" * 90)
print("05.2 — GRID SEARCH  (Stratified 5-Fold, scoring=roc_auc)")
print("-" * 90)
print("  Esto puede tardar varios minutos...\n")

modelos        = {}   # mejor pipeline reentrenado por modelo
mejores_params = {}   # registro de los parametros ganadores

for nombre, pipeline_base in base_modelos.items():
    gs = GridSearchCV(
        estimator  = pipeline_base,
        param_grid = param_grid[nombre],
        cv         = cv,          # Stratified 5-Fold definido en 05.1
        scoring    = 'roc_auc',
        n_jobs     = -1,
        refit      = True,        # re-entrena con best params sobre todo Xtr
        verbose    = 0,
    )
    gs.fit(Xtr, ytr)
    modelos[nombre]        = gs.best_estimator_
    mejores_params[nombre] = gs.best_params_
    print(f"  {nombre:22}  CV-AUC optimo: {gs.best_score_*100:.1f}%")
    for param, val in gs.best_params_.items():
        print(f"    {param.replace('clf__',''):30} = {val}")
    print()

print("  Grid Search completado. Mejores parametros en 'mejores_params'.")

# ---------------------------------------------------------------------------
# 05.3 — Funcion de entrenamiento y evaluacion
# ---------------------------------------------------------------------------
def evaluar(nombre, modelo, usar_peso):
    """Entrena, calcula CV-AUC y metricas en test. usar_peso aplica
    sample_weight (los 3 modelos lo aceptan)."""
    t0 = time.time()

    # --- CV-AUC sobre el train (deteccion de leakage) ---
    cv_auc = cross_val_score(modelo, Xtr, ytr, cv=cv,
                             scoring='roc_auc', n_jobs=-1).mean()

    # --- fit final ---
    if usar_peso:
        # sample_weight va al ultimo paso del pipeline
        try:
            modelo.fit(Xtr, ytr, clf__sample_weight=wtr)
        except (TypeError, ValueError):
            modelo.fit(Xtr, ytr)
    else:
        modelo.fit(Xtr, ytr)

    # --- prediccion en test ---
    proba = modelo.predict_proba(Xte)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    # AUC: ponderado si corresponde
    sw = wte if usar_peso else None
    test_auc = roc_auc_score(yte, proba, sample_weight=sw)
    acc  = accuracy_score(yte, pred, sample_weight=sw)
    tn, fp, fn, tp = confusion_matrix(yte, pred, sample_weight=sw).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        'modelo': nombre,
        'cv_auc': cv_auc,
        'test_auc': test_auc,
        'brecha_pp': (cv_auc - test_auc) * 100,
        'accuracy': acc,
        'sensibilidad': sens,
        'especificidad': spec,
        'segundos': time.time() - t0,
    }

# ---------------------------------------------------------------------------
# 05.4 — ENTRENAMIENTO A: modelos SIN ponderar (comparable con Etiopia)
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("A) MODELOS SIN PONDERAR  (comparables con el paper de Etiopia)")
print("-" * 90)

res_sin = []
modelos_fit = {}     # modelos ya entrenados (sin ponderar) para el Paso 06
for nombre, modelo in modelos.items():
    p = pipe(modelo.named_steps['clf'])
    r = evaluar(nombre, p, usar_peso=False)
    res_sin.append(r)
    modelos_fit[nombre] = p          # p quedo entrenado dentro de evaluar()
    print(f"  {nombre:22} CV-AUC {r['cv_auc']*100:5.1f}  "
          f"Test-AUC {r['test_auc']*100:5.1f}  "
          f"brecha {r['brecha_pp']:+5.1f}pp  "
          f"Sens {r['sensibilidad']*100:4.1f}  Esp {r['especificidad']*100:4.1f}")

df_sin = pd.DataFrame(res_sin).set_index('modelo')

# ---------------------------------------------------------------------------
# 05.5 — ENTRENAMIENTO B: modelos PONDERADOS (representativo de Peru)
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("B) MODELOS PONDERADOS con peso muestral HV005  (representativo de Peru)")
print("-" * 90)

res_pon = []
for nombre, modelo in modelos.items():
    r = evaluar(nombre, pipe(modelo.named_steps['clf']), usar_peso=True)
    res_pon.append(r)
    print(f"  {nombre:22} CV-AUC {r['cv_auc']*100:5.1f}  "
          f"Test-AUC {r['test_auc']*100:5.1f}  "
          f"brecha {r['brecha_pp']:+5.1f}pp  "
          f"Sens {r['sensibilidad']*100:4.1f}  Esp {r['especificidad']*100:4.1f}")

df_pon = pd.DataFrame(res_pon).set_index('modelo')

# ---------------------------------------------------------------------------
# 05.6 — Diagnostico de leakage
# ---------------------------------------------------------------------------
print("\n" + "=" * 90)
print("DIAGNOSTICO DE BRECHA CV-TEST  (|brecha| < 5pp = sin leakage)")
print("=" * 90)
for nombre in modelos:
    b = df_sin.loc[nombre, 'brecha_pp']
    estado = "OK" if abs(b) < 5 else "REVISAR"
    print(f"  {nombre:22} brecha {b:+5.1f}pp  -> {estado}")

# ---------------------------------------------------------------------------
# 05.7 — Mejor modelo
# ---------------------------------------------------------------------------
mejor = df_sin['test_auc'].idxmax()
print("\n" + "=" * 90)
print(f"MEJOR MODELO (sin ponderar): {mejor}")
print(f"  Test-AUC      : {df_sin.loc[mejor,'test_auc']*100:.1f}%")
print(f"  CV-AUC        : {df_sin.loc[mejor,'cv_auc']*100:.1f}%")
print(f"  Brecha        : {df_sin.loc[mejor,'brecha_pp']:+.1f}pp")
print(f"  Sensibilidad  : {df_sin.loc[mejor,'sensibilidad']*100:.1f}%")
print(f"  Especificidad : {df_sin.loc[mejor,'especificidad']*100:.1f}%")
print(f"  Version ponderada Test-AUC: {df_pon.loc[mejor,'test_auc']*100:.1f}%")
print("=" * 90)

print("\n  Tablas disponibles: df_sin (sin ponderar) y df_pon (ponderado)")
print("  PASO 05 COMPLETADO.")

# =============================================================================
# SERIALIZACIÓN DEL PIPELINE — Predicción de Anemia Infantil
# Agregar al final del PASO 05 en tu notebook de Colab
# Guarda el pipeline completo: Imputer + Scaler + Random Forest
# =============================================================================

import joblib

# El pipeline de Random Forest ya entrenado (sin ponderar)
# 'modelos_fit' fue construido en el Paso 05 del pipeline v3
pipeline_rf = modelos_fit["Random Forest"]

# Guardar el modelo serializado
joblib.dump(pipeline_rf, "modelo_anemia_rf.pkl")
print("✅ Modelo serializado: modelo_anemia_rf.pkl")

# Guardar también la lista de predictores (orden exacto requerido)
import json
with open("predictores.json", "w") as f:
    json.dump(PREDICTORES, f, ensure_ascii=False, indent=2)
print(f"✅ Lista de {len(PREDICTORES)} predictores guardada: predictores.json")

# Verificación rápida
pipeline_cargado = joblib.load("modelo_anemia_rf.pkl")
proba_test = pipeline_cargado.predict_proba(Xte[:5])[:, 1]
print(f"✅ Verificación OK — primeras 5 probabilidades: {[f'{p:.2%}' for p in proba_test]}")

# Descargar desde Colab
try:
    from google.colab import files
    files.download("modelo_anemia_rf.pkl")
    files.download("predictores.json")
    print("⬇️  Descarga iniciada para ambos archivos.")
except Exception:
    print("(Fuera de Colab: archivos guardados en el directorio de trabajo.)")


##########################################################################################
# =============================================================================
# PIPELINE — PASO 06: EVALUACION FINAL, COMPARACION Y GRAFICOS
# (VERSION 3 MODELOS)
# Ejecutar DESPUES del Paso 05 (necesita df_sin, df_pon, modelos_fit, Xtr, Xte,
# ytr, yte, PREDICTORES, df_modelo).
#
# CONTENIDO:
#  06.1  Tabla comparativa final (sin ponderar vs ponderado)
#  06.2  Comparacion con el estudio de Etiopia (Yimer et al. 2025)
#  06.3  Grafico 1: curvas ROC de los 3 modelos
#  06.4  Grafico 2: importancia de variables (Random Forest)
#  06.5  Grafico 3: matriz de confusion del mejor modelo
#  06.6  Descarga de los graficos
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix

print("=" * 90)
print("PASO 06 — EVALUACION FINAL Y GRAFICOS  (3 MODELOS)")
print("=" * 90)

assert 'df_sin' in dir(), "Ejecuta primero el Paso 05"

# ---------------------------------------------------------------------------
# 06.1 — Tabla comparativa final
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("06.1 — TABLA COMPARATIVA (Peru ENDES 2024)")
print("-" * 90)

orden = df_sin['test_auc'].sort_values(ascending=False).index
print(f"\n  {'Modelo':22} {'AUC s/pond':>11} {'AUC pond':>10} {'Sens':>7} {'Esp':>7} {'Brecha':>8}")
for m in orden:
    print(f"  {m:22} {df_sin.loc[m,'test_auc']*100:10.1f}% "
          f"{df_pon.loc[m,'test_auc']*100:9.1f}% "
          f"{df_sin.loc[m,'sensibilidad']*100:6.1f}% "
          f"{df_sin.loc[m,'especificidad']*100:6.1f}% "
          f"{df_sin.loc[m,'brecha_pp']:+7.1f}pp")

mejor = df_sin['test_auc'].idxmax()
print(f"\n  Mejor modelo: {mejor}")

# ---------------------------------------------------------------------------
# 06.2 — Comparacion con Etiopia
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("06.2 — COMPARACION CON ETIOPIA (Yimer et al. 2025)")
print("-" * 90)

# AUC reportado en el paper de Etiopia (EDHS 2016) — solo los 3 modelos usados
etiopia = {
    'Random Forest':        81.8,
    'Decision Tree':        56.3,
    'Logistic Regression':  56.1,
}

print(f"\n  {'Modelo':22} {'Peru AUC':>10} {'Etiopia AUC':>13} {'Dif.':>9}")
for m in orden:
    pe = df_sin.loc[m, 'test_auc'] * 100
    et = etiopia.get(m, np.nan)
    dif = pe - et
    print(f"  {m:22} {pe:9.1f}% {et:12.1f}% {dif:+8.1f}pp")

print(f"""
  LECTURA:
   - Etiopia RF 81.8% se obtuvo con SMOTE fuera del CV (leakage):
     su brecha CV-Test rondaba ~5pp.
   - Peru RF {df_sin.loc[mejor,'test_auc']*100:.1f}% tiene brecha {df_sin.loc[mejor,'brecha_pp']:+.1f}pp (honesto).
   - La diferencia no es "peor modelo": es la diferencia entre una
     metrica inflada y una metrica valida.
""")

# ---------------------------------------------------------------------------
# 06.3 — GRAFICO 1: Curvas ROC de los 3 modelos
# ---------------------------------------------------------------------------
print("-" * 90)
print("06.3 — Grafico 1: curvas ROC")
print("-" * 90)

plt.figure(figsize=(8, 7))
for m, modelo in modelos_fit.items():
    proba = modelo.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(yte, proba)
    plt.plot(fpr, tpr, lw=2,
             label=f"{m} (AUC {auc(fpr,tpr)*100:.1f}%)")
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Azar (50%)')
plt.xlabel('1 - Especificidad (Falsos positivos)')
plt.ylabel('Sensibilidad (Verdaderos positivos)')
plt.title('Curvas ROC — Prediccion de anemia infantil (ENDES 2024)')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('grafico_1_curvas_roc.png', dpi=300, bbox_inches='tight')
plt.show()
print("  Guardado: grafico_1_curvas_roc.png")

# ---------------------------------------------------------------------------
# 06.4 — GRAFICO 2: Importancia de variables (Random Forest)
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("06.4 — Grafico 2: importancia de variables (Random Forest)")
print("-" * 90)

rf = modelos_fit["Random Forest"].named_steps['clf']
importancias = pd.Series(rf.feature_importances_, index=PREDICTORES)
importancias = importancias.sort_values(ascending=True)

plt.figure(figsize=(8, 9))
plt.barh(importancias.index, importancias.values, color='#1D9E75')
plt.xlabel('Importancia (reduccion media de impureza)')
plt.title('Importancia de variables — Random Forest')
plt.tight_layout()
plt.savefig('grafico_2_importancia.png', dpi=300, bbox_inches='tight')
plt.show()
print("  Guardado: grafico_2_importancia.png")

print(f"\n  Top 10 predictores mas importantes:")
for v, imp in importancias.sort_values(ascending=False).head(10).items():
    print(f"    {v:24} {imp*100:5.1f}%")

# ---------------------------------------------------------------------------
# 06.5 — GRAFICO 3: Matriz de confusion del mejor modelo
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("06.5 — Grafico 3: matriz de confusion (mejor modelo)")
print("-" * 90)

proba_mejor = modelos_fit[mejor].predict_proba(Xte)[:, 1]
pred_mejor = (proba_mejor >= 0.5).astype(int)
cm = confusion_matrix(yte, pred_mejor)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Greens')
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha='center', va='center',
                 fontsize=14,
                 color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.xticks([0, 1], ['Sin anemia', 'Con anemia'])
plt.yticks([0, 1], ['Sin anemia', 'Con anemia'])
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.title(f'Matriz de confusion — {mejor}')
plt.colorbar()
plt.tight_layout()
plt.savefig('grafico_3_matriz_confusion.png', dpi=300, bbox_inches='tight')
plt.show()
print("  Guardado: grafico_3_matriz_confusion.png")

# ---------------------------------------------------------------------------
# 06.6 — Descarga de los graficos
# ---------------------------------------------------------------------------
print("\n" + "-" * 90)
print("06.6 — Descarga de archivos")
print("-" * 90)
try:
    from google.colab import files
    for g in ['grafico_1_curvas_roc.png',
              'grafico_2_importancia.png',
              'grafico_3_matriz_confusion.png']:
        files.download(g)
    print("  Descarga iniciada para los 3 graficos.")
except Exception:
    print("  (Fuera de Colab: los PNG quedaron en el directorio de trabajo.)")

print("\n" + "=" * 90)
print("PASO 06 COMPLETADO — Pipeline finalizado (3 modelos).")
print("=" * 90)